In [1]:

from torch.utils.data import Dataset
import json
from tqdm import tqdm
class JsonlDataset(Dataset):
    def __init__(self, path: str):
        self.path = path
        self.data = []
        with open(path, "r") as f:
            for line in tqdm(f, desc="Loading data"):
                self.data.append(json.loads(line))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = JsonlDataset("/traindata/maksim/repos/blog_private/torch_distributed/06_retrieval_embeddings_finetune/data/train.jsonl")
example = train_dataset[0]
example

Loading data: 400782it [00:12, 31336.04it/s]


{'query': ')what was the immediate impact of the success of the manhattan project?',
 'pos_doc': ['Introduction',
  'The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.'],
 'neg_doc': [['-',
   'Uploaded on Sep 19, 2009. This video shows one way to manage calendar spread trade that is going in the wrong direction. This video highlights the use of the thinkorswim analyze tab to determine the impact of adjusting the trade. This video provided by http://www.success-with-options.com.'],
  ['The World Clock - Time Zone difference from USA â Kansas â Manhattan',
   'The World Clock-Time Zone difference from U.S.A. – Kansas – Manhattan.dd or subtract the given number of hours to/from Manhattan time to get the time in these 

In [3]:
import argparse
import os
import tqdm
import torch
from typing import List, Union, Optional, Tuple, Mapping, Dict

from contextlib import nullcontext
from torch.utils.data import DataLoader
from functools import partial
from datasets import load_dataset
from typing import Dict, List
from transformers.file_utils import PaddingStrategy
from transformers import (
    AutoTokenizer,
    AutoModel,
    PreTrainedTokenizerFast,
    DataCollatorWithPadding,
    HfArgumentParser,
    BatchEncoding
)

def move_to_cuda(sample):
    if len(sample) == 0:
        return {}

    def _move_to_cuda(maybe_tensor):
        if torch.is_tensor(maybe_tensor):
            return maybe_tensor.cuda(non_blocking=True)
        elif isinstance(maybe_tensor, dict):
            return {key: _move_to_cuda(value) for key, value in maybe_tensor.items()}
        elif isinstance(maybe_tensor, list):
            return [_move_to_cuda(x) for x in maybe_tensor]
        elif isinstance(maybe_tensor, tuple):
            return tuple([_move_to_cuda(x) for x in maybe_tensor])
        elif isinstance(maybe_tensor, Mapping):
            return type(maybe_tensor)({k: _move_to_cuda(v) for k, v in maybe_tensor.items()})
        else:
            return maybe_tensor

    return _move_to_cuda(sample)




def _psg_transform_func(tokenizer: PreTrainedTokenizerFast, args,
                        examples: Dict[str, List]) -> BatchEncoding:
    batch_dict = tokenizer(examples['title'],
                           text_pair=examples['contents'],
                           max_length=args.p_max_len,
                           padding=PaddingStrategy.DO_NOT_PAD,
                           truncation=True)
    # for co-Condenser reproduction purpose only
    if args.model_name_or_path.startswith('Luyu/'):
        del batch_dict['token_type_ids']

    return batch_dict


/traindata/maksim/miniconda3/envs/lightning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset = load_dataset('json', data_files="msmarco_bm25_official/passages.jsonl.gz")['train']
dataset[0]

Generating train split: 8841823 examples [00:17, 514050.42 examples/s]


{'id': '0',
 'contents': 'The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.',
 'title': 'Introduction'}

In [ ]:


model_name = "intfloat/simlm-base-msmarco"

tokenizer: PreTrainedTokenizerFast = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()
model.cuda()

In [ ]:
dataset.set_transform(partial(_psg_transform_func, tokenizer, args))